## 8. `ClaudeAgentOptions`：唯一的配置总线

> 来源：[Agent SDK reference - Python](https://code.claude.com/docs/en/agent-sdk/python)

`query()` 与 `ClaudeSDKClient` 的 `options` 参数收的就是这个 dataclass，不传时等价于 `ClaudeAgentOptions()` 全默认（`query()` 签名见 §5「最小可运行示例」）。所有能力都从它挂进去。先看一个把七组功能各挑一两个代表字段的构造示例，再按组展开（每组括注本笔记展开的章节）：

In [ ]:
options = ClaudeAgentOptions(
    # ① 工具与权限
    allowed_tools=["Read", "Glob", "Grep"],  # 自动批准名单
    disallowed_tools=[
        "Bash(rm *)"
    ],  # "Bash(rm *)" 只 deny 匹配的调用；写 "Bash"（不带括号）则该 tool 定义整个不发给模型
    permission_mode="default",
    # ② 系统提示与模型
    system_prompt={"type": "preset", "preset": "claude_code", "append": "回答用中文。"},
    model="claude-sonnet-4-5",
    effort="medium",
    # ③ 会话
    resume=None,  # 传旧 session_id 即恢复该会话（§13）
    # ④ 运行环境
    cwd="/path/to/repo",
    env={"ENABLE_TOOL_SEARCH": "auto"},
    setting_sources=[],  # [] = 不读磁盘上任何 Claude 配置文件（CLAUDE.md、settings.json 等），行为完全由本对象决定
    # ⑤ 扩展
    mcp_servers={"state": state_server},
    hooks={"PreToolUse": [HookMatcher(matcher="Bash", hooks=[audit_hook])]},
    agents={"reviewer": AgentDefinition(description="...", prompt="...")},
    # ⑥ 输出形态
    include_partial_messages=False,  # True 则流里多出 StreamEvent（§17「流式输出」）
    # ⑦ 预算与轮次
    max_turns=20,
    max_budget_usd=1.0,
)


不用背全部字段，按**七组功能**记：

**① 工具与权限**（§9、§10）
- `allowed_tools: list[str]`：自动批准名单
- `disallowed_tools: list[str]`：黑名单，两种写法效果不同——写不带括号的工具名 `"Bash"`，该 tool 的定义整个不发给模型，模型不知道它存在；写带范围的 `"Bash(rm *)"`，tool 定义仍在 context 里，只有匹配该模式的调用被 deny
- `tools: list[str] | preset`：**可用性名单**——只有列出的内置 tool 进 context（与 allowed_tools 是两层概念，见 §10.4「tools vs allowed_tools」）
- `permission_mode` / `can_use_tool` / `permission_prompt_tool_name`：权限模式 / 审批回调 / 自定义审批 MCP tool
- `sandbox: SandboxSettings`：命令沙箱（§21.5「sandbox」）

**② 系统提示与模型**（§19）
- `system_prompt`：字符串，或 preset dict `{"type": "preset", "preset": "claude_code", "append": "..."}`；**不设时用的是只覆盖 tool calling 的最小 prompt，不是 Claude Code 完整人格**
- `model` / `fallback_model`：模型 ID / 主模型失败时的兜底
- `effort`：`"low" | "medium" | "high" | "xhigh" | "max"`，用延迟和 token 换推理深度
- `thinking: ThinkingConfig`：extended thinking 配置（和 effort 各管各的、可分别设置；旧字段 `max_thinking_tokens` 已废弃）

**③ 会话**（§13）
- `resume` / `continue_conversation` / `fork_session`：恢复指定 session / 接最近一次 / 分叉
- `session_store` / `session_store_flush`：外部存储 adapter / 刷写策略（`"batched"` 每 turn 一次，`"eager"` 每帧）
- `enable_file_checkpointing`：文件改动快照，可回滚（§21.4「文件快照与任务清单」）

**④ 运行环境**
- `cwd` / `add_dirs`：工作目录 / 额外可访问目录
- `env`：合并进 CLI 子进程的环境变量（`ENABLE_TOOL_SEARCH`、`CLAUDE_CONFIG_DIR`、OTel、超时重试等开关都从这里进）
- `setting_sources`：选择读取哪几层磁盘配置文件。三层按"对谁生效 × 在哪生效"区分——
  - `"user"` = 个人 × 所有项目（`~/.claude/settings.json`）；
  - `"project"` = 团队 × 本仓库（仓库 `.claude/settings.json`、CLAUDE.md，随 git 共享）；
  - `"local"` = 个人 × 本仓库（`.claude/settings.local.json`，gitignore 不入库）。
  **省略 = 三层全加载（与 CLI 行为一致）**，`[]` = 一层都不读，agent 行为完全由代码里的 options 决定（§20「setting_sources」）
- `settings` / `cli_path` / `extra_args` / `max_buffer_size` / `stderr` / `user`：settings 文件路径 / CLI 路径 / 透传 CLI 参数 / stdout 缓冲上限 / stderr 回调 / 用户标识

**⑤ 扩展**
- `mcp_servers`：外部/进程内 MCP（§10、§12）；`strict_mcp_config=True` 时只认这里传的，无视 `.mcp.json`、用户配置和 claude.ai connectors
- `hooks`：生命周期钩子（§11「Hooks」）
- `agents: dict[str, AgentDefinition]`：子 agent（§14）
- `skills` / `plugins`：技能过滤 / 插件加载（§20「setting_sources」）
- `betas: list[SdkBeta]`：beta 特性开关

**⑥ 输出形态**
- `include_partial_messages`：开启后流里多出 `StreamEvent`（§17「流式输出」）
- `output_format`：`{"type": "json_schema", "schema": {...}}` 结构化输出（§18）
- `include_hook_events`：把 hook 生命周期事件透出到消息流

**⑦ 预算与轮次**
- `max_turns`：最多 tool-use 轮次
- `max_budget_usd`：客户端成本估算达到该美元数即停（与 `total_cost_usd` 同一口径，见 §21.2「成本追踪」）

> 💡 `allowed_tools` 是"自动批准"而非"能用的全集"——不在名单里的工具**不是不能用，是会触发审批**；真正控制"哪些工具存在"的是 `tools` 字段。这组概念分不清是新手第一大坑，§9「权限系统与 HITL」和 §10「自定义工具」展开。